In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [6]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [13]:
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [7]:
# Encode categorical variable for gender
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [8]:
# one hot encode geography
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [9]:
onehot_encoder_geo.categories_

[array(['France', 'Germany', 'Spain'], dtype=object)]

In [10]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [14]:
# combine OHE data with original data
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [15]:
#split data into feature and target
x=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

In [16]:
#split data in training and testing
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)


#sclae these features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [17]:
# save encoders and label for later use
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('label_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [21]:
### ANN regression problem statement
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [22]:
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1) #output layer no need to give activation function for regression problem it will automatically pick linear regression
])

# compile the model
model.compile(optimizer='adam',loss = 'mean_absolute_error',metrics=['mae'])
model.summary()

/Users/prao000/Egift files/Course/GENAI/Genai/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

log_dir = "regressionlogs/fit"+ datetime.datetime.now().strftime("%Y%m%d - %H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)


In [27]:
# early stopping
early_stoppping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [28]:
#train model
history = model.fit(
    X_train,y_train,
    validation_data = (X_test,y_test),
    epochs = 100,
    callbacks = [early_stoppping_callback,tensorboard_callback] 
    )

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 99461.0078 - mae: 99461.0078 - val_loss: 98465.9141 - val_mae: 98465.9141
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 858us/step - loss: 98405.8438 - mae: 98405.8438 - val_loss: 96686.6406 - val_mae: 96686.6406
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 883us/step - loss: 98028.8516 - mae: 98028.8516 - val_loss: 92137.0000 - val_mae: 92137.0000
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 888us/step - loss: 92914.1016 - mae: 92914.1016 - val_loss: 84685.2109 - val_mae: 84685.2109
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 804us/step - loss: 82838.9453 - mae: 82838.9453 - val_loss: 75422.5312 - val_mae: 75422.5312
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 797us/step - loss: 74574.9531 - mae: 74574.9531 - val_loss: 66339.5391 - val_mae: 66339.5391
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 816us/step - loss: 65051.0859 - mae: 65051.0859 - val_loss: 59001.8711 - val_mae: 59001.8711
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s

In [29]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir regressionlogs --port 6009


In [34]:
#Evaluate model
test_loss,test_mae = model.evaluate(X_test,y_test)
print(f"Test mae is:{test_mae}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 836us/step - loss: 51103.8438 - mae: 51103.8438
Test mae is:50257.9453125


In [35]:
model.save('regression_model.h5')